# CC9. Big Data. Extracting a story from millions of prices.



UK price microdata: provided in class

In [9]:
import pandas as pd
import altair as alt

In [10]:
prices_df = pd.read_csv('../data/UKMicroData/db_prices.csv')
prices_df.sample(5)

,quote_date,shop_code,item_id_raw,region,price,indicator_box,item_id
39633294,199611.0,77.0,520220,3.0,7.59,NaN,520220
521111,202211.0,951.0,210204,10.0,0.95,NaN,210204
4780746,200008.0,51.0,211701,8.0,0.26,NaN,211701
43053304,201009.0,66.0,620315,6.0,3.00,NaN,620315
41447518,199305.0,917.0,610211,9.0,24.50,NaN,610211


In [11]:
items_df = pd.read_csv('../data/UKMicroData/db_item_clean.csv')
items_df

,item_id,description,date_quote_s,date_quote_e,n_obs
0,210101,LARGE LOAF-WHITE-SLICED-800G,198802,200401,36039
1,210102,LARGE LOAF-WHITE-UNSLICED-800G,198802,202409,56048
2,210105,LARGE WHOLEMEAL LOAF-UNSLICED,198802,200301,27161
3,210106,SIX BREAD ROLLS-WHITE/BROWN,198802,202409,65710
4,210107,"BROWN LOAF,400G,SLICED-GRAN",198903,200401,29361
...,...,...,...,...,...
1381,640233,LEISURE CENTRE MEMBERSHIP,200302,201201,17695
1382,640240,LIVERY CHARGES PER WEEK,200802,202409,20235
1383,640243,SOFT PLAY SESSION TIME PERIOD,201802,202409,9280
1384,640244,CLIMBING WALL SESSION,202202,202409,1526


<br>

---

Searching for items we want.

- Filter for only current items (`date_quote_e` is latest data)
- Filter for only food items (`item_id` starts with '2')

In [12]:
# Filter to food items
food_df = items_df[items_df['item_id'].astype(str).str.startswith('2')].copy()
food_df

,item_id,description,date_quote_s,date_quote_e,n_obs
0,210101,LARGE LOAF-WHITE-SLICED-800G,198802,200401,36039
1,210102,LARGE LOAF-WHITE-UNSLICED-800G,198802,202409,56048
2,210105,LARGE WHOLEMEAL LOAF-UNSLICED,198802,200301,27161
3,210106,SIX BREAD ROLLS-WHITE/BROWN,198802,202409,65710
4,210107,"BROWN LOAF,400G,SLICED-GRAN",198903,200401,29361
...,...,...,...,...,...
426,220325,VENDING MACHINE - SOFT DRINK,200702,202301,18450
427,220326,TAKEAWAY CHICKEN & CHIPS,201202,202409,18287
428,220327,T'AWAY COOKED SAVOURY PASTRY,201802,202409,17164
429,220328,BURGER IN A BUN EAT IN OR TAKE,202002,202409,16644


So we have 207 items that are currently tracked in the CPI basket.  

---

### Comparing oil prices before and after Russia's invassion of Ukraine

In [13]:
match = ['SPREADABLE BUTTER', 'BUTTER 250G SPEC COO', 'MARGARINE/LOW FAT SPREAD-500G', 'OLIVE OIL - 500ML - 1 LITRE']

# pattern from the list of products
pattern = '|'.join(match)

# Filter the DataFrame
food_filtered = food_df[food_df['description'].str.contains(pattern, case=False)]

food_filtered

,item_id,description,date_quote_s,date_quote_e,n_obs
131,211305,SPREADABLE BUTTER,201302,202409,32303
132,211306,BUTTER 250G SPEC COO,201302,202409,32627
136,211407,MARGARINE/LOW FAT SPREAD-500G,199602,202301,46329
137,211408,OLIVE OIL - 500ML - 1 LITRE,200702,202409,46952


Before we build the chart, we should check when the series started for each of these ingredients.

So the earliest point where each item has data is  199902, so we'll filter for this later.

In [23]:
# Filter for my items
items = [211305, 211306, 211407, 211408]
df = prices_df[prices_df['item_id'].isin(items)].copy()

# Filter for after 199902
df = df[df['quote_date'] >= 201301].copy()

# Parse into a nice date format.
df['quote_date'] = pd.to_datetime(df['quote_date'], format="%Y%m")

# Let's aggregate across regions, by finding the mean.
df = df.groupby(['quote_date', 'item_id'])['price'].mean().reset_index()

df

,quote_date,item_id,price
0,2013-01-01,211407,1.628643
1,2013-01-01,211408,3.331472
2,2013-02-01,211305,2.736530
3,2013-02-01,211306,1.491762
4,2013-02-01,211407,1.649565
...,...,...,...
537,2024-08-01,211306,2.130517
538,2024-08-01,211408,8.447991
539,2024-09-01,211305,3.256485
540,2024-09-01,211306,2.161870


Now, we need to scale the prices to match our ingredient amounts.

For example, our spaghetti prices are measured by 500g. Since we want for 450g, we should multiply by 450/500.



In [24]:
# Define the scales for each.
scales = {
    211305: 1,
    211306: 4,
    211407: 2,
    211408: 1
    }

# Map scales to item_id and apply scaling
df['scale_factor'] = df['item_id'].map(scales)  # Map scales
df['scaled_price'] = df['price'] * df['scale_factor']  # Apply scaling

# Divide to get per portion
df['price_portion'] = df['scaled_price'] / 6

- Matches the item_id values with the corresponding scale factors in the scales dictionary.
- Creates a new column scale_factor containing the scale factor for each row.

In [ ]:
df

Finally, let's add labels for each of the series.

In [25]:
## Define a lookup between item ids and labels we want to add.
labels = {
    211305: 'Spreadable butter',
    211306: 'Special butter',
    211407: 'Low-fat Margarine',
    211408: 'Olive oil'
}
df['labels'] = df['item_id'].map(labels)

In [ ]:
df.head()

### Plotting 

---

#### 9b Russian invasion on UK prices

In [54]:
base = alt.Chart(df).mark_line().encode(
    alt.X('quote_date:T', axis=alt.Axis(title='Date')),
    alt.Y('price:Q', axis=alt.Axis(title='Price per Kg or Lt (£)')),
    alt.Color('labels:N', title='Product', legend=alt.Legend(orient='bottom', columns=2, 
                                                            labelFontSize=15, titleFontSize=20))
).properties(
    width=500,  # Adjust the width here
    height=400,  # Adjust the height here
    title={
        "text": "Russian invasion and UK prices",
        "subtitle": "Sample of high-exposed products",
        "fontSize": 20,
        "subtitleFontSize": 15
    }
)

# Vertical dashed line for Brexit
vline = alt.Chart(pd.DataFrame({'quote_date': ['2020-01-31']})).mark_rule(color='red', strokeDash=[5, 5]).encode(
    x='quote_date:T'
)

# Text "Brexit"
text = alt.Chart(pd.DataFrame({'quote_date': ['2020-01-31'], 'label': ['Brexit']})).mark_text(
    align='left',
    baseline='top',
    dx=-40,  # Position of text
    dy=-30,  # Position of text
    fontWeight='bold'  # Bold text
).encode(
    x='quote_date:T',
    text='label'
)

# Vertical dashed line for the Ukraine Russian War 
vline2 = alt.Chart(pd.DataFrame({'quote_date': ['2022-02-24']})).mark_rule(color='red', strokeDash=[5,5]).encode(
    x='quote_date:T'
)

# Text "Ukraine-Russia War"
text2 = alt.Chart(pd.DataFrame({'quote_date': ['2022-02-24'], 'label': ['Ukraine-Russia War']})).mark_text(
    align='left',
    baseline='top',
    dx=-110,  # Position of text
    dy=-160,  # Position of text
    fontWeight='bold'  # Bold text
).encode(
    x='quote_date:T',
    text='label'
)


# Combine the base chart and the vertical line
chart = base + vline + vline2 + text2 + text

chart.save('../figures/figure9.json')

chart

alt.LayerChart(...)

---

### 9a Impact of Brexit on food prices (aggregated)

In [27]:
# Filter to just rows where end date is latest (max) date.
food_df_max = food_df[food_df['date_quote_e'] == food_df['date_quote_e'].max()].copy()
#items_df.head()


# Making breakfast

In [28]:
search_terms = ['cereal', 'milk', 'coffee', 'bread']
breakfast_items = [210106, 211709, 210213, 213006] # I am gonna pick up this items.

breakfast_df = food_df_max[food_df_max['description'].str.contains('|'.join(search_terms), case=False)].copy()

breakfast_df = food_df_max[food_df_max['item_id'].isin(breakfast_items)].copy()


breakfast_df

,item_id,description,date_quote_s,date_quote_e,n_obs
3,210106,SIX BREAD ROLLS-WHITE/BROWN,198802,202409,65710
24,210213,BREAKFAST CEREAL 1,200602,202409,49321
156,211709,SHOP MILK-WHOLE MILK-4PT/2LTR,199002,202409,88004
375,213006,COFFEE PODS PACK 8-16,201602,202409,17064


In [29]:
# Filter for my items
items = breakfast_df['item_id']

df2 = prices_df[prices_df['item_id'].isin(items)].copy()

# Filter for after 199902
df2 = df2[df2['quote_date'] >= 201602].copy()

# Parse into a nice date format.
df2['quote_date'] = pd.to_datetime(df2['quote_date'], format="%Y%m")

# Let's aggregate across regions, by finding the mean.
df2 = df2.groupby(['quote_date', 'item_id'])['price'].mean().reset_index()

# For each date and item, find the mean average price
df2 = df2.groupby(['quote_date', 'item_id'])['price'].mean()

# Reset the index to get back to standard 'date', 'region', 'price' columns
df2 = df2.reset_index()

df2.sample(5)

,quote_date,item_id,price
341,2023-03-01,210213,2.462559
348,2023-05-01,210106,1.205504
316,2022-09-01,210106,1.141628
209,2020-06-01,210213,2.525404
291,2022-02-01,213006,3.694329


In [30]:
df2.head()

df2['item_id'].value_counts() # Balanced panel

item_id
210106    104
210213    104
211709    104
213006    104
Name: count, dtype: int64

In [31]:
# Define the scales for each.
scales2 = {
    210106: 1/6 , # 1 bread roll
    211709: 1/10 , # 200 ml of whole milk
    210213: 1/10, # 1/10 of the cereal pack 
    213006: 1/16 # 1 pod of coffee
    }

# Map scales to item_id and apply scaling
df2['scale_factor'] = df2['item_id'].map(scales2)  # Map scales
df2['scaled_price'] = df2['price'] * df2['scale_factor']  # Apply scaling

# Divide to get per portion
#df2['price_portion'] = df2['scaled_price'] / 6

In [192]:

df2.sample(5)

,quote_date,item_id,price,scale_factor,scaled_price
243,2021-02-01,213006,3.912933,0.0625,0.244558
274,2021-10-01,211709,1.266750,0.1000,0.126675
19,2016-06-01,213006,3.907607,0.0625,0.244225
85,2017-11-01,210213,2.228270,0.1000,0.222827
130,2018-10-01,211709,1.193592,0.1000,0.119359


#### Plotting

In [55]:
# Your existing label lookup and data mapping
label_lookup = {
    210106: '1 Bread roll',
    211709: '200ml of Whole Milk',
    210213: '1 Portion of cereal',
    213006: '1 cup of coffee'
}

df2['labels'] = df2['item_id'].map(label_lookup)

# Define a custom color palette
custom_palette = ['grey', 'teal', 'gold', 'lightblue']  # Example colors

# Build the chart again, using 'labels' for the color encoding now.
base2 = alt.Chart(df2).mark_area().encode(
    alt.X('quote_date:T', axis=alt.Axis(title='Date')),
    alt.Y('scaled_price:Q', axis=alt.Axis(title='Price (£)')),
    alt.Color('labels:N', title='Product', legend=alt.Legend(orient='bottom', columns=2,  labelFontSize=15, titleFontSize=20))
        ).properties(
        width=500,  # Adjust the width here
        height=400,  # Adjust the height here
    title={
        "text": "It's still better to eat breakfast at home",
        "subtitle": "Average price of a breakfast in the UK",
        "fontSize": 20,
        "subtitleFontSize": 15
    }
)

# Vertical dashed line for Brexit
vline = alt.Chart(pd.DataFrame({'quote_date': ['2020-01-31']})).mark_rule(color='red', strokeDash=[5, 5]).encode(
    x='quote_date:T'
)

# Vertical dashed line for Brexit
vline = alt.Chart(pd.DataFrame({'quote_date': ['2020-01-31']})).mark_rule(color='red', strokeDash=[5, 5]).encode(
    x='quote_date:T'
)

# Text "Brexit"
text = alt.Chart(pd.DataFrame({'quote_date': ['2020-01-31'], 'label': ['Brexit']})).mark_text(
    align='left',
    baseline='top',
    dx=-40,  # Position of text
    dy=-160,  # Position of text
    fontWeight='bold'  # Bold text
).encode(
    x='quote_date:T',
    text='label'
)

# Vertical dashed line for the Ukraine Russian War 
vline2 = alt.Chart(pd.DataFrame({'quote_date': ['2022-02-24']})).mark_rule(color='red', strokeDash=[5,5]).encode(
    x='quote_date:T'
)

# Text "Ukraine-Russia War"
text2 = alt.Chart(pd.DataFrame({'quote_date': ['2022-02-24'], 'label': ['Ukraine-Russia War']})).mark_text(
    align='left',
    baseline='top',
    dx=5,  # Position of text
    dy=-180,  # Position of text
    fontWeight='bold'  # Bold text
).encode(
    x='quote_date:T',
    text='label'
)

# Combine the base chart and the vertical line
chart2 = base2 + vline + vline2 + text + text2

chart2.save('../figures/figure9b.json')

chart2

alt.LayerChart(...)